# Связь активности субталамического ядра с двигательными симптомами при болезни Паркинсона

На примере одной синтетической записи показано, как рассчитываются нормированный ранг PPC и выраженность бета-ритма в LFP. Затем по сводным таблицам исследования строятся графики фазовой привязки и её связи с двигательными симптомами. Отдельно показана связь бета-активности самого LFP с теми же симптомами.

## Как вычисляется нормированный ранг

Если из 500 сдвигов 450 дали меньшую PPC, а 50 — большую, наблюдаемая PPC находится примерно на 90-м процентиле. В коде применяется небольшая конечновыборочная поправка. При совпадениях учитывается половина равных значений.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from demo_phase_rank import synthetic_recording, phase_rank, beta_over_background
from plot_results import group_figure, clinic_figure

rank = (450 + 0.5) / (500 + 1)
print(f"Нормированный ранг в примере: {rank:.3f}")

Нормированный ранг в примере: 0.899


## Два показателя на синтетической записи

Частота колебания медленно меняется в пределах бета-полосы. Вероятность спайка увеличивается около гребня. Первая мера сравнивает согласованность фаз спайков со сдвигами, вторая оценивает превышение мощности над фоном по спектру LFP. Значения ниже иллюстрируют расчёт для одной записи.

In [2]:
lfp, spike_times, fs = synthetic_recording()
ppc, rank = phase_rank(lfp, spike_times, fs, np.random.default_rng(10))
beta = beta_over_background(lfp, fs)
print(f"PPC = {ppc:.4f}, нормированный ранг = {rank:.3f}")
print(f"Бета в LFP над фоном = {beta:.3f} в шкале log10 мощности")

PPC = 0.1279, нормированный ранг = 0.999
Бета в LFP над фоном = 0.779 в шкале log10 мощности


## Фазовая привязка на реальных данных

После удаления четырёх повторов осталось 372 нейрона из 183 файлов у 22 пациентов. Таблица содержит результат анализа среднего нормированного ранга с иерархическим бутстрепом. В столбце `q_fdr` приведены p-значения после поправки FDR на множественные сравнения по четырём частотным диапазонам..

In [3]:
results = Path("results")
group = pd.read_csv(results / "group.csv")
print(group[["band", "mean_rank", "ci_low", "ci_high", "p_boot", "q_fdr"]].to_string(index=False))
group_figure()

 band  mean_rank  ci_low  ci_high  p_boot  q_fdr
 Тета      0.481   0.441    0.522  0.3518 0.3518
Альфа      0.535   0.485    0.590  0.1740 0.2320
 Бета      0.574   0.521    0.632  0.0072 0.0288
Гамма      0.551   0.496    0.608  0.0728 0.1456


![Средние ранги по диапазонам](results/group_rank.png)

Групповой средний ранг в бете равен 0,574. На уровне отдельных нейронов фазовая привязка в бете значима после FDR у 13 нейронов из 372.

## Сопоставление с симптомами

Для каждого полушария по нейронам усредняли обычный ранг, по файлам брали медиану бета-активности LFP над фоном. Каждый показатель сопоставляли с двумя клиническими исходами. Таблица содержит коэффициенты Спирмена из исходного анализа и p до поправки. Число пациентов отличается из-за порога в три нейрона для фазовой привязки и доступности клиники ON.

In [4]:
clinic = pd.read_csv(results / "clinic.csv")
print(clinic.to_string(index=False))
clinic_figure()

   measure       outcome  hemispheres  patients   rho  ci_low  ci_high     p
phase_rank contra_ar_off           29        18  0.16   -0.23     0.50 0.503
phase_rank contra_ar_pct           29        18 -0.02   -0.40     0.39 0.944
  lfp_beta contra_ar_off           35        22  0.04   -0.34     0.39 0.865
  lfp_beta contra_ar_pct           34        21  0.06   -0.32     0.43 0.789


![Четыре клинические корреляции](results/phase_and_clinic.png)

Для фазовой привязки спайков корреляция с суммарным баллом брадикинезии и ригидности в OFF составила ρ = +0,16, а с относительным улучшением этих симптомов после приёма леводопы — ρ = −0,02.

Для выраженности бета-ритма в LFP корреляция с суммарным баллом в OFF составила ρ = +0,04, а с относительным улучшением после приёма леводопы — ρ = +0,06.

Во всех четырёх случаях 95%-й доверительный интервал включал ноль. В этой выборке ни один из двух показателей не показал статистически значимой связи с тяжестью симптомов или их улучшением после приёма леводопы.